# 3. MIMIC-IV: download, cohort, labels, waveforms, five-fold design and conversion to the EchoNext format

**Replication cohort.** It is assembled from four MIMIC-IV modules (Beth Israel Deaconess Medical Center) by linking ECG waveforms, structured echocardiography measurements, and hospital and emergency department records:

* adults with an ECG recorded 0–365 days before a transthoracic echocardiogram (TTE) that reported a numeric LVEF and at least one valve finding; each eligible ECG is labelled from the first eligible TTE that followed it;
* echocardiograms at or after the first record of a valve prosthesis, repair or transcatheter device are not eligible;
* ECGs whose machine report states pacing or poor signal quality, or whose waveform has missing samples or a flat lead, are excluded.

Steps of this notebook: (3.2) cohort and labels, (3.3) waveforms, (3.4) final cohort and five-fold design, (3.5) characteristics (Table 1, Supplementary Table 2), (3.6) constants for converting the waveforms to the EchoNext format (used in notebook 04).

## 3.1 Download the data (once, by hand)

MIMIC-IV, MIMIC-IV-ED and MIMIC-IV-Echo are **credentialed** PhysioNet projects: you need a credentialed PhysioNet account (identity check and the CITI "Data or Specimens Only Research" course) and must sign each project's data use agreement. MIMIC-IV-ECG is **open access**. Run the commands in a terminal **inside `DATA_DIR`**; `-x` keeps the PhysioNet folder structure, which gives the default paths of notebook 00.

| Module (version) | Page | Files used |
|---|---|---|
| MIMIC-IV (3.1) | https://physionet.org/content/mimiciv/3.1/ | `hosp/admissions.csv.gz`, `hosp/patients.csv.gz` |
| MIMIC-IV-ED (2.2) | https://physionet.org/content/mimic-iv-ed/2.2/ | `ed/edstays.csv.gz` |
| MIMIC-IV-Echo (1.0.1) | https://physionet.org/content/mimic-iv-echo/1.0.1/ | `structured-measurement.csv.gz` (the DICOM images are **not** needed) |
| MIMIC-IV-ECG (1.0) | https://physionet.org/content/mimic-iv-ecg/1.0/ | `record_list.csv`, `machine_measurements.csv`, `files/` (WFDB records) |

```bash
U=<your PhysioNet username>
wget -x -N -c --user $U --ask-password https://physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz https://physionet.org/files/mimiciv/3.1/hosp/patients.csv.gz
wget -x -N -c --user $U --ask-password https://physionet.org/files/mimic-iv-ed/2.2/ed/edstays.csv.gz
wget -x -N -c --user $U --ask-password https://physionet.org/files/mimic-iv-echo/1.0.1/structured-measurement.csv.gz
wget -r -N -c -np https://physionet.org/files/mimic-iv-ecg/1.0/          # open access, 90 GB (or download the 34 GB ZIP from the project page)
```

To save space, download only `record_list.csv` and `machine_measurements.csv` of MIMIC-IV-ECG first, run section 3.2, and then download only the records listed in `WORK_DIR/mimic/needed_ecg_records.txt` (about 27 GB) with the command printed at the end of section 3.2.

In [ ]:
%run ./00_settings.ipynb
import re
from concurrent.futures import ThreadPoolExecutor
from numpy.lib.format import open_memmap

FILES = {"admissions": MIMIC_IV_DIR / "hosp" / "admissions.csv.gz", "patients": MIMIC_IV_DIR / "hosp" / "patients.csv.gz",
         "edstays": MIMIC_ED_DIR / "ed" / "edstays.csv.gz", "echo_measurements": MIMIC_ECHO_DIR / "structured-measurement.csv.gz",
         "ecg_records": MIMIC_ECG_DIR / "record_list.csv", "ecg_machine": MIMIC_ECG_DIR / "machine_measurements.csv"}
for k, p in FILES.items():
    print(f"{k:<18} {'ok     ' if p.exists() else 'MISSING'} {p}")
assert all(p.exists() for p in FILES.values()), "download the missing files first (section 3.1)"
OUT = results("03_mimic")
WM = work("mimic")

## 3.2 Cohort and labels

### Echocardiography

`structured-measurement.csv.gz` is in long format (one row per measurement). The fields below are pivoted to one row per study. Numeric values outside plausible ranges are set to missing. Every label is **positive** for a grade of moderate or greater (or the numeric threshold), **negative** for a definite lower grade, and **not determinable** otherwise; a blank field counts as negative in the analysis, as in EchoNext and in a previous external validation of EchoNext-Mini in MIMIC-IV.

* LVEF: the reported value (midpoint of a reported range), otherwise the biplane or three-dimensional estimate; LVEF ≤45%.
* Wall thickness: the larger of the septal and inferolateral thickness; ≥1.3 cm.
* Aortic stenosis: reader grade of moderate or greater, peak velocity ≥3.0 m/s, mean gradient ≥20 mm Hg, or valve area ≤1.5 cm².
* Mitral and tricuspid regurgitation ≥ moderate (2+); right ventricular dysfunction: moderate or severe hypokinesis; other components as listed.
* Prosthesis, repair or transcatheter device: any prosthetic-valve field of the second reporting system, or a matching value of the leaflet fields.

In [ ]:
RELIABLE_MIN, RELIABLE_MAXDIFF = 85.0, 5.0
EF_VARS = ["lvef", "lvef_upper", "biplane_lvef", "lvef_3d"]
NUMV = EF_VARS + ["septal_thickness", "inf_lat_thickness", "av_pk_vel", "av_mean_grad", "av_area_continuity", "tr_velocity", "tr_mmhg"]
BOUNDS = {"lvef": (5, 90), "lvef_upper": (5, 90), "biplane_lvef": (5, 90), "lvef_3d": (5, 90), "septal_thickness": (0.3, 4.0),
          "inf_lat_thickness": (0.3, 4.0), "av_pk_vel": (0.3, 8.0), "av_mean_grad": (0, 150), "av_area_continuity": (0.1, 6.0),
          "tr_velocity": (0.5, 7.0), "tr_mmhg": (0, 150)}
VALVE = ["aortic_stenosis", "aortic_regurg", "mitral_regurg", "tricuspid_regurg", "pulm_regurg"]
CATV = VALVE + ["rv_function", "pericardial_effusion", "phtn_severity", "av_leaflets", "mv_leaflets", "tv_leaflets", "pt_status"]
PROSTH_B = ["avr_structure1", "avr_structure2", "mvr_structure1", "mvr_structure2", "mvr_structure3",
            "tvr_structure1", "tvr_structure2", "tvr_structure3", "pvr_structure1", "pvr_structure2"]
MARK_B = "tapse"                                   # a field recorded only by the second reporting system
PROSTH_RE = re.compile(r"prosthe|mechanical|sapien|corevalve|evolut|lotus|homograft|tilting disk|ball and cage|annular ring|"
                       r"annuloplasty|mitraclip|triclip|pascal|tmvr|ttvr|cardiaaq|^bileaflet$", re.I)
PACE_DROP = re.compile(r"wandering atrial pacemaker|pacer detection suspended[^|]*", re.I)
PACE_RE = re.compile(r"pac(?:ed|ing)\b|pacemaker|\bpacer\b", re.I)
QUAL_RE = re.compile(r"unsuitable|data quality|poor quality|lead reversal|leads reversed|electrode reversal|reversed leads|missing|"
                     r"technical error|technically unsatisfactory|not enough leads|external noise|not used for morphology", re.I)
GRADES = {   # categorical grades: (positive = moderate or greater, definite negative); every other value is not determinable
    "aortic_stenosis": ({"Mod (1.0-1.5cm2)", "Severe (<1.0cm2)", "Very severe (>5m/s;>60mmHg)", "Severe (<0.6cm2/m2)",
                         "Low flow/low gradient severe", "Mod to severe (0.8-1.0cm2)"},
                        {"Mild (>1.5cm2)", "Minimal", "None, incr vel high output", "None, incr grad from AR/stroke vol"}),
    "mitral_regurg": ({"Mod [2+]", "Moderate-severe (3+)", "Severe (4+)"}, {"Trivial", "Mild (1+)", "Mild-moderate (1-2+)", "Physiologic"}),
    "tricuspid_regurg": ({"Mod [2+]", "Mod-severe [3+]", "Severe [4+]"}, {"Trivial", "Mild [1+]", "Mild-mod [1-2+]", "Physiologic"}),
    "aortic_regurg": ({"Mod [2+]", "Mod-severe [3+]", "Severe [4+]"}, {"Trace", "Mild [1+]", "Mild-mod [1-2+]"}),
    "pulm_regurg": ({"Significant", "Mod/severe", "Severe"}, {"Physiologic", "Mild"}),
    "rv_function": ({"Moderate global RV hypo", "Severe global hypo"},
                    {"Nl RV function", "Mild global RV hypo", "Low normal function", "Apical free wall hypo", "Basal RV hypo (McConnell's sign)", "Hyperdynamic"}),
    "phtn_severity": ({"Mod (TR 37-60mmHg)", "Severe (TR >60mmHg)", "Mod-severe (50-60mmHg)"},
                      {"Normal (TR<26mmHg)", "Mild (TR 26-36mmHg)", "Borderline (TR 26-30mmHg)", "HIgh normal (TR 21-25 mmHg)", "Mild-mod (TR 35-40mmHg)",
                       "PA HTN unlikely"}),
    "pericardial_effusion": ({"Moderate (1.0-2.0cm)", "Large (>2.0cm)", "Moderate-large"},
                             {"Trivial", "Very small (<0.5cm)", "Small (<1.0cm)", "Small-moderate", "Anterior fat pad"}),
}
CAT_LABEL = {"mr_modplus": "mitral_regurg", "tr_modplus": "tricuspid_regurg", "ar_modplus": "aortic_regurg", "pr_modplus": "pulm_regurg",
             "rv_modplus": "rv_function", "ph_modplus": "phtn_severity", "pe_modplus": "pericardial_effusion"}


def to_num(s):
    """Numbers, or the first number in strings such as '>75', '<20' or '55-60'."""
    x = pd.to_numeric(s, errors="coerce")
    odd = x.isna() & s.notna()
    if odd.any():
        x[odd] = pd.to_numeric(s[odd].str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")
    return x


def load_echo():
    """-> study table (subject, time, test type, reporting system) and the wide table of the needed fields (one row per study)."""
    need = set(NUMV + CATV + PROSTH_B + [MARK_B])
    parts, studies = [], []
    for ch in pd.read_csv(FILES["echo_measurements"], chunksize=3_000_000, low_memory=False,
                          usecols=["subject_id", "measurement_id", "measurement_datetime", "test_type", "measurement", "result"],
                          dtype={"result": str, "measurement": str}):
        studies.append(ch.drop_duplicates("measurement_id")[["measurement_id", "subject_id", "measurement_datetime", "test_type"]])
        parts.append(ch.loc[ch.measurement.isin(need), ["measurement_id", "measurement", "result"]])
    st = pd.concat(studies).drop_duplicates("measurement_id").set_index("measurement_id")
    st["t_echo"] = pd.to_datetime(st.measurement_datetime, errors="coerce").astype("datetime64[ns]")
    k = pd.concat(parts)
    sys_b = set(k.loc[k.measurement == MARK_B, "measurement_id"])
    k["result"] = k.result.fillna("").str.strip()
    k = k[k.result != ""]
    Wd = k.drop_duplicates(["measurement_id", "measurement"]).pivot(index="measurement_id", columns="measurement", values="result")
    Wd = Wd.reindex(st.index).reindex(columns=sorted(need))
    st["system"] = np.where(st.index.isin(sys_b), "B", "A")
    return st, Wd


def study_table(st, Wd):
    """Structure variables, the 11 labels with their 'determinable' masks, valve findings and prostheses, one row per study."""
    num = {}
    for c in NUMV:
        v = to_num(Wd[c])
        lo, hi = BOUNDS[c]
        num[c] = v.mask(v.notna() & ((v < lo) | (v > hi)))
    num = pd.DataFrame(num)
    S = pd.DataFrame(index=Wd.index)
    rng_ = num.lvef_upper.notna() & num.lvef.notna() & (num.lvef_upper >= num.lvef)
    ef = num.lvef.where(~rng_, (num.lvef + num.lvef_upper) / 2)                 # reported range -> midpoint
    S["ef"] = ef.fillna(num.biplane_lvef).fillna(num.lvef_3d)
    S["sept"], S["infl"] = num.septal_thickness, num.inf_lat_thickness
    S["wall"] = S[["sept", "infl"]].max(axis=1)
    S["tr_grad"], S["trv"] = num.tr_mmhg, num.tr_velocity
    lab, det = {}, {}
    lab["lvef_le45"], det["lvef_le45"] = S.ef <= 45, S.ef.notna()
    lab["lvwt_ge13"], det["lvwt_ge13"] = S.wall >= 1.3, S.wall.notna()
    pos, neg = GRADES["aortic_stenosis"]
    asg = Wd.aortic_stenosis
    as_num = (num.av_pk_vel >= 3.0) | (num.av_mean_grad >= 20) | (num.av_area_continuity <= 1.5)
    any_num = num[["av_pk_vel", "av_mean_grad", "av_area_continuity"]].notna().any(axis=1)
    lab["as_modplus"], det["as_modplus"] = asg.isin(pos) | as_num, asg.isin(pos | neg) | any_num
    for key, col in CAT_LABEL.items():
        pos, neg = GRADES[col]
        lab[key], det[key] = Wd[col].isin(pos), Wd[col].isin(pos | neg)
    lab["trv_ge32"], det["trv_ge32"] = S.trv >= 3.2, S.trv.notna()
    for k in M_COMPONENTS:
        S[k], S["det_" + k] = lab[k].to_numpy(bool), det[k].to_numpy(bool)
    S["valve_any"] = Wd[VALVE].notna().any(axis=1) | num.av_pk_vel.notna()
    pro = Wd[PROSTH_B].notna().any(axis=1)
    for c in ("av_leaflets", "mv_leaflets", "tv_leaflets", "aortic_regurg"):
        pro |= Wd[c].fillna("").str.contains(PROSTH_RE)
    S["prosth"] = pro.to_numpy(bool)
    S["echo_inpatient"] = (Wd.pt_status == "Inpatient").to_numpy()
    return S


t0 = time.time()
st, Wd = load_echo()
S = study_table(st, Wd)
tte = (st.test_type == "tte").to_numpy()
t_pro = st.loc[S.prosth.to_numpy() & st.t_echo.notna()].groupby("subject_id").t_echo.min()      # first record of a prosthesis or repair
before_pro = (st.t_echo < st.subject_id.map(t_pro).fillna(pd.Timestamp.max)).to_numpy()
E0 = tte & S.ef.notna().to_numpy()                 # TTE with a numeric LVEF
E1 = E0 & S.valve_any.to_numpy()                   # ... and at least one valve finding
E2 = E1 & before_pro                               # ... before any prosthesis or repair
print(f"echocardiography: {len(st):,} studies; TTE {tte.sum():,}; with LVEF {E0.sum():,}; + valve finding {E1.sum():,}; "
      f"+ no prior prosthesis {E2.sum():,} ({time.time() - t0:.0f} s)")

### ECGs, race and ethnicity, clinical setting

* ECG machine reports: a pacing statement (except "wandering atrial pacemaker" and "pacer detection suspended") or a signal-quality statement excludes the ECG. The automated measurements (ventricular rate, PR interval, QRS duration, QTc) feed the measurement comparator of notebook 05.
* Race and ethnicity are recorded at every admission and emergency visit. Values naming Hispanic, Latino or South American origin are classed as Hispanic; each patient is assigned the category, other than unknown, recorded most often.
* Clinical setting of the ECG: emergency department stay, otherwise hospital admission, otherwise outpatient, from the time stamps.

In [ ]:
def race_category(v):
    """Recorded MIMIC-IV race value -> category (Hispanic ethnicity takes precedence, as in EchoNext)."""
    s = str(v).upper()
    if "HISPANIC" in s or "LATINO" in s or "SOUTH AMERICAN" in s:
        return "hispanic"
    if s.startswith("WHITE") or s == "PORTUGUESE":
        return "white"
    if s.startswith("BLACK"):
        return "black"
    if s.startswith("ASIAN"):
        return "asian"
    if s in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "NAN", ""):
        return "unknown"
    return "other"


def patient_race_table():
    """Most frequent informative category per patient over admissions and ED stays (unknown only if nothing else is recorded)."""
    r = pd.concat([pd.read_csv(FILES["admissions"], usecols=["subject_id", "race"]), pd.read_csv(FILES["edstays"], usecols=["subject_id", "race"])])
    r["cat"] = r.race.astype(str).map({v: race_category(v) for v in r.race.astype(str).unique()})
    cnt = r.groupby(["subject_id", "cat"]).size().reset_index(name="n")
    cnt["informative"] = (cnt.cat != "unknown").astype(int)
    cnt = cnt.sort_values(["subject_id", "informative", "n", "cat"], ascending=[True, False, False, True])
    out = cnt.drop_duplicates("subject_id").set_index("subject_id")[["cat"]]
    out["n_inf"] = cnt[cnt.informative == 1].groupby("subject_id").size().reindex(out.index).fillna(0).astype(int)
    return out


def load_ecg():
    """ECG records with the pacing and quality flags and the automated measurements of the machine report."""
    rl = pd.read_csv(FILES["ecg_records"])
    mm = pd.read_csv(FILES["ecg_machine"], low_memory=False).drop_duplicates("study_id")
    rep = [c for c in mm.columns if c.startswith("report_")]
    txt = mm[rep[0]].fillna("").astype(str)
    for c in rep[1:]:
        txt = txt + " | " + mm[c].fillna("").astype(str)
    paced = txt.str.replace(PACE_DROP, "", regex=True).str.contains(PACE_RE)
    qual = txt.str.contains(QUAL_RE)
    ok = lambda v, lo, hi: v.where((v >= lo) & (v <= hi))
    rr, po, qo, qe, te = (mm[c].astype(float) for c in ("rr_interval", "p_onset", "qrs_onset", "qrs_end", "t_end"))
    rr = ok(rr, 200, 3000)
    M = pd.DataFrame({"study_id": mm.study_id, "paced": paced.to_numpy(), "qual": qual.to_numpy(), "ventricular_rate": 60000 / rr,
                      "pr_interval": ok(qo.where(qo < 29999) - po.where(po < 29999), 40, 500),
                      "qrs_duration": ok(qe.where(qe < 29999) - qo.where(qo < 29999), 30, 400),
                      "qt_corrected": ok(te.where(te < 29999) - qo.where(qo < 29999), 150, 900) / np.sqrt(rr / 1000)})
    E = rl.merge(M, on="study_id", how="left")
    E["paced"], E["qual"] = E.paced.fillna(False).astype(bool), E.qual.fillna(False).astype(bool)
    E["t_ecg"] = pd.to_datetime(E.ecg_time, errors="coerce").astype("datetime64[ns]")
    return E


def within(E, tab, start, end):
    """Was each ECG recorded during one of the patient's stays in `tab` (between `start` and `end`)?"""
    t = tab[["subject_id", start, end]].copy()
    t[start] = pd.to_datetime(t[start], errors="coerce").astype("datetime64[ns]")
    t[end] = pd.to_datetime(t[end], errors="coerce").astype("datetime64[ns]")
    t = t.dropna().sort_values(start)
    m = pd.merge_asof(E[["subject_id", "t_ecg"]].reset_index().sort_values("t_ecg"), t, left_on="t_ecg", right_on=start, by="subject_id", direction="backward")
    return pd.Series((m.t_ecg <= m[end]).to_numpy(), index=m["index"]).reindex(E.index).fillna(False).to_numpy(bool)


def first_echo_after(E, st, elig):
    """First eligible TTE 0-365 days after each ECG (same patient)."""
    e = st.loc[elig & st.t_echo.notna(), ["subject_id", "t_echo"]].reset_index().sort_values("t_echo")
    m = pd.merge_asof(E.loc[E.t_ecg.notna(), ["subject_id", "t_ecg"]].reset_index().sort_values("t_ecg"), e, left_on="t_ecg", right_on="t_echo",
                      by="subject_id", direction="forward", tolerance=pd.Timedelta(days=365))
    return m.set_index("index")[["measurement_id", "t_echo"]].reindex(E.index)

### Linking, exclusions and the derivation flow (Supplementary Fig. 1)

In [ ]:
t0 = time.time()
E = load_ecg()
flow = []
step = lambda name, m: flow.append({"step": name, "ecgs": int(m.sum()), "patients": int(E.subject_id[m].nunique())})
keep = np.ones(len(E), bool); step("MIMIC-IV-ECG records", keep)
pat = pd.read_csv(FILES["patients"], usecols=["subject_id", "gender", "anchor_age", "anchor_year", "anchor_year_group"])
E = E.merge(pat, on="subject_id", how="left")
E["age"] = np.minimum(E.anchor_age + (E.t_ecg.dt.year - E.anchor_year), 90)
keep &= (E.t_ecg.notna() & (E.age >= 18) & E.gender.isin(["F", "M"])).to_numpy(); step("Adults with recorded sex and ECG time", keep)
for name, el in (("ECG 0–365 days before TTE with numeric LVEF", E0), ("TTE with at least one valve finding", E1),
                 ("No prior valve prosthesis or repair", E2)):
    mt = first_echo_after(E, st, pd.Series(el, index=st.index))
    step(name, keep & mt.measurement_id.notna().to_numpy())
keep &= mt.measurement_id.notna().to_numpy()                       # linked to the first eligible TTE (the last match above)
E["measurement_id"], E["t_echo"] = mt.measurement_id, mt.t_echo
keep &= ~E.paced.to_numpy(); step("No pacing statement in machine report", keep)
keep &= ~E.qual.to_numpy(); step("No signal-quality statement in machine report", keep)

C = E[keep].sort_values("path").reset_index(drop=True)              # candidate ECGs, in file-path order (= waveform array order)
C["measurement_id"] = C.measurement_id.astype(st.index.dtype)
C["row"] = np.arange(len(C))
C["gap_days"] = (C.t_echo - C.t_ecg).dt.total_seconds() / 86400
C["male"] = (C.gender == "M").astype(int)
race = patient_race_table()
C["race6"] = C.subject_id.map(race.cat).fillna("unknown")
C["race_multi"] = C.subject_id.map(race.n_inf).fillna(0).astype(int) > 1
adm = pd.read_csv(FILES["admissions"], usecols=["subject_id", "admittime", "dischtime"])
ed = pd.read_csv(FILES["edstays"], usecols=["subject_id", "intime", "outtime"])
in_ed, in_adm = within(C, ed, "intime", "outtime"), within(C, adm, "admittime", "dischtime")
C["setting"] = np.where(in_ed, "emergency", np.where(in_adm, "inpatient", "outpatient"))
yg = C.anchor_year_group.astype(str).str.slice(0, 4).astype(float) + 1                    # middle of the three-year anchor group
C["year_approx"] = yg + (C.t_echo.dt.year - C.anchor_year)
C["system"] = st.system.reindex(C.measurement_id).to_numpy()
Sx = S.reindex(C.measurement_id).reset_index(drop=True)
for c in Sx.columns:
    C[c] = Sx[c].to_numpy()
print(pd.DataFrame(flow).to_string(index=False))
print(f"{len(C):,} candidate ECGs of {C.subject_id.nunique():,} patients ({time.time() - t0:.0f} s)")

### Which components enter the analysis? (Supplementary Table 2)

Many structured fields are blank, and a blank field treated as normal could create spurious differences between groups. A component is used only if it can be determined in at least 85% of studies **in every race and ethnicity group and in each of the two reporting systems**, with differences between groups below 5 percentage points (computed on the TTE linked to each patient's most recent candidate ECG). The composite of the main analysis (`shd_main`) is any of the components that pass. `shd_full` (any of all 11 components) defines "no recorded SHD" in notebook 06.

In [ ]:
lat_c = C.sort_values(["t_ecg", "study_id"]).drop_duplicates("subject_id", keep="last")
rows = []
for k in M_COMPONENTS:
    d = lat_c["det_" + k]
    r = {"label_key": k, "label": LABEL_NAME[k], "overall": 100 * d.mean()}
    for gname in RACE4:
        r[gname] = 100 * d[lat_c.race6 == gname].mean()
    for s in ("A", "B"):
        r[f"system_{s}"] = 100 * d[lat_c.system == s].mean()
    r["max_diff_race"] = max(r[gname] for gname in RACE4) - min(r[gname] for gname in RACE4)
    r["min_any"] = min([r[gname] for gname in RACE4] + [r["system_A"], r["system_B"]])
    r["reliable"] = bool(r["min_any"] >= RELIABLE_MIN and r["max_diff_race"] < RELIABLE_MAXDIFF)
    rows.append(r)
asc = pd.DataFrame(rows)
asc.round(2).to_csv(OUT / "ascertainment.csv", index=False, encoding="utf-8-sig")
main_comp = [k for k in M_COMPONENTS if asc.set_index("label_key").reliable[k]]
C["shd_main"] = C[main_comp].any(axis=1)
C["shd_full"] = C[M_COMPONENTS].any(axis=1)
for k in M_COMPONENTS + ["shd_main", "shd_full"]:
    C[k] = C[k].astype(float)
labels = {"main": ["shd_main"] + main_comp, "main_components": main_comp, "not_used": [k for k in M_COMPONENTS if k not in main_comp],
          "rule": {"min_pct_every_group_and_system": RELIABLE_MIN, "max_between_group_difference_pp": RELIABLE_MAXDIFF}}
save_json(labels, OUT / "labels.json")
save_json({"flow": flow}, OUT / "flow.json")
keepc = ["row", "subject_id", "study_id", "path", "t_ecg", "measurement_id", "t_echo", "gap_days", "age", "male", "race6", "race_multi", "setting",
         "system", "year_approx", "echo_inpatient", "ef", "sept", "infl", "wall", "tr_grad", "trv", "ventricular_rate", "pr_interval", "qrs_duration",
         "qt_corrected", "shd_main", "shd_full"] + M_COMPONENTS + ["det_" + k for k in M_COMPONENTS]
C[keepc].to_parquet(WM / "cohort_candidates.parquet", index=False)
(WM / "needed_ecg_records.txt").write_text("\n".join(f"https://physionet.org/files/mimic-iv-ecg/1.0/{p}.{x}" for p in C.path for x in ("hea", "dat")) + "\n")
print("labels of the analysis:", labels["main"])
print("To download only the needed ECG records (inside DATA_DIR):  wget -x -N -c -i", WM / "needed_ecg_records.txt")
asc[["label", "overall", "white", "black", "hispanic", "asian", "system_A", "system_B", "max_diff_race", "reliable"]].round(1)

## 3.3 Waveforms

Every MIMIC-IV-ECG record has 12 leads × 5000 samples at 500 Hz, stored as 16-bit integers at 200 units per mV (lead order I, II, III, aVR, aVF, aVL, V1–V6 in the files). The records of the candidate ECGs are read into one contiguous array `WORK_DIR/mimic/waveforms_int16.npy` (N × 12 × 5000, about 26 GB) in the EchoNext lead order (I, II, III, aVR, aVL, aVF, V1–V6). A record fails the check if its header is unexpected, a sample is missing (−32768) or a lead is flat.

Reading takes about 15 minutes from a hard disk and resumes where it stopped if it is interrupted.

In [ ]:
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
N_SAMP, FS_MIMIC, GAIN = 5000, 500, 200.0
WAVE, WAVE_QC, PROGRESS = WM / "waveforms_int16.npy", WM / "waveform_qc.parquet", WM / "waveforms_progress.json"


def read_record(path):
    """-> (status, 12 x 5000 int16 in the EchoNext lead order or None, missing samples, flat leads)."""
    try:
        lines = [l for l in (MIMIC_ECG_DIR / (path + ".hea")).read_text().splitlines() if l and not l.startswith("#")]
        f = lines[0].split()
        sig = [l.split() for l in lines[1:]]
        names = [s[-1] for s in sig]
        if int(f[1]) != 12 or int(float(f[2])) != FS_MIMIC or int(f[3]) != N_SAMP or len(sig) != 12:
            return "layout", None, 0, 0
        if any(s[1] != "16" for s in sig) or set(names) != set(LEADS):
            return "layout", None, 0, 0
        gains = [float(s[2].split("(")[0].split("/")[0]) for s in sig]
        base = [float(s[2].split("(")[1].split(")")[0]) if "(" in s[2] else float(s[4]) for s in sig]
        raw = np.fromfile(MIMIC_ECG_DIR / (path + ".dat"), dtype="<i2")
        if raw.size != 12 * N_SAMP:
            return "size", None, 0, 0
        x = raw.reshape(N_SAMP, 12).T[[names.index(l) for l in LEADS]]
        n_miss = int((x == -32768).sum())
        if any(gn != GAIN for gn in gains) or any(bs != 0 for bs in base):          # rescale to 200 units per mV with baseline 0
            order = [names.index(l) for l in LEADS]
            gg, bb = np.array(gains)[order, None], np.array(base)[order, None]
            x = np.clip(np.rint((x.astype(np.float64) - bb) / gg * GAIN), -32767, 32767).astype(np.int16)
        n_flat = int((x.max(1) == x.min(1)).sum())
        return ("ok" if n_miss == 0 and n_flat == 0 else "missing" if n_miss else "flat"), np.ascontiguousarray(x), n_miss, n_flat
    except FileNotFoundError:
        return "nofile", None, 0, 0
    except Exception as e:                                                      # noqa: BLE001 - counted, never printed per record
        return "error:" + type(e).__name__, None, 0, 0


def read_all_waveforms(paths, chunk=2000, threads=8):
    n = len(paths)
    status = np.full(n, "", dtype=object); n_miss = np.zeros(n, np.int64); n_flat = np.zeros(n, np.int16)
    start = 0
    if WAVE.exists() and PROGRESS.exists() and load_json(PROGRESS).get("n") == n:       # resume an interrupted run
        start = load_json(PROGRESS)["done"]
        old = pd.read_parquet(WM / "waveform_qc_partial.parquet")
        status[:start], n_miss[:start], n_flat[:start] = old.status[:start], old.n_miss[:start], old.n_flat[:start]
    Wv = open_memmap(WAVE, mode="r+" if start else "w+", dtype=np.int16, shape=(n, 12, N_SAMP))
    t0 = time.time()
    with ThreadPoolExecutor(threads) as ex:
        for s in range(start, n, chunk):
            for i, (stt, x, nm, nf) in enumerate(ex.map(read_record, paths[s:s + chunk])):
                status[s + i], n_miss[s + i], n_flat[s + i] = stt, nm, nf
                if x is not None:
                    Wv[s + i] = x
            done = min(s + chunk, n)
            if (s // chunk) % 5 == 0 or done == n:
                Wv.flush()
                pd.DataFrame({"status": status[:done], "n_miss": n_miss[:done], "n_flat": n_flat[:done]}).to_parquet(WM / "waveform_qc_partial.parquet")
                save_json({"n": n, "done": done}, PROGRESS)
                print(f"{done:,}/{n:,} records, {(done - start) / max(time.time() - t0, 1e-9):.0f} per second", flush=True)
    Wv.flush(); del Wv
    return status, n_miss, n_flat


Cc = pd.read_parquet(WM / "cohort_candidates.parquet", columns=["row", "path"])
if WAVE_QC.exists() and len(pd.read_parquet(WAVE_QC)) == len(Cc):
    print("waveforms already read:", WAVE)
else:
    status, n_miss, n_flat = read_all_waveforms(Cc.path.tolist())
    qc = pd.DataFrame({"row": Cc.row.to_numpy(), "status": status, "n_miss": n_miss, "n_flat": n_flat})
    qc["wave_ok"] = qc.status == "ok"
    qc.to_parquet(WAVE_QC, index=False)
qc = pd.read_parquet(WAVE_QC)
waveform_summary = {"n_candidates": int(len(qc)), "status_counts": {k: int(v) for k, v in qc.status.value_counts().items()}, "n_ok": int(qc.wave_ok.sum())}
save_json(waveform_summary, OUT / "waveform_summary.json")
waveform_summary

## 3.4 Final cohort and five-fold design

The final cohort is the candidate ECGs whose waveform passed the check. Each patient is evaluated on the most recent ECG. The five-fold design is built exactly as for EchoNext (notebook 02): folds stratified by race and ethnicity (six categories) × the composite on the most recent ECG, validation = 15% of the other patients.

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

C = load_mimic_cohort()
lat = C[C.mr]
assert lat.subject_id.is_unique and len(lat) == C.subject_id.nunique()
pats = lat.subject_id.to_numpy()
strata = (lat.race6.astype(str) + "_" + lat[M_COMP].astype(int).astype(str)).to_numpy()
fold_of = np.empty(len(pats), int)
for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED_FOLDS).split(pats, strata)):
    fold_of[te] = k
role = np.zeros((N_FOLDS, len(pats)), np.int8)
for k in range(N_FOLDS):
    rest = np.flatnonzero(fold_of != k)
    _, va = next(StratifiedShuffleSplit(1, test_size=VAL_FRAC, random_state=SEED_FOLDS + 1 + k).split(rest, strata[rest]))
    role[k, rest[va]] = 1
    role[k, fold_of == k] = 2
D = {"pi_of_row": C.subject_id.map(pd.Series(np.arange(len(pats)), index=pats)).to_numpy(), "mr_row": C.mr.to_numpy(),
     "emb_row": C.row.to_numpy(), "lat_idx": lat.index.to_numpy(), "fold_of": fold_of, "role": role}
np.savez_compressed(WM / "design.npz", **D)
g_lat = race_index(lat.race6)
final = {"n_patients": int(len(lat)), "n_ecgs": int(len(C)), "group_patients": {r: int((g_lat == i).sum()) for i, r in enumerate(RACE4)},
         "folds": [{"fold": k, "fit_patients": int((role[k] == 0).sum()), "val_patients": int((role[k] == 1).sum()), "eval_patients": int((role[k] == 2).sum()),
                    "fit_ecgs": int(len(fold_rows(D, k)[0]))} for k in range(N_FOLDS)]}
save_json(final, OUT / "final_cohort.json")
print(f"final cohort: {final['n_ecgs']:,} ECGs, {final['n_patients']:,} patients;", final["group_patients"])
pd.DataFrame(final["folds"])

## 3.5 Characteristics (Table 1) and prevalence by group (Fig. 3c,d)

In [ ]:
main = mimic_labels()
lat = lat.reset_index(drop=True)
rows = []
for name, d, dall in [("All", lat, C)] + [(gname, lat[lat.race6 == gname], C[C.race6 == gname]) for gname in GROUPS6]:
    r = {"cohort": "MIMIC-IV", "group": name, "patients": len(d), "ecgs": len(dall), "age": median_iqr(d.age), "female_pct": 100 * (1 - d.male.mean())}
    for s in ("inpatient", "emergency", "outpatient"):
        r[f"{s}_pct"] = 100 * (d.setting == s).mean()
    for lab in main:
        r["prev_" + LABEL_NAME[lab]] = 100 * d[lab].mean()
        r["pos_" + LABEL_NAME[lab]] = int(d[lab].sum())
    r["gap_days"] = median_iqr(d.gap_days, 1)                                     # days from the ECG to its TTE
    rows.append(r)
characteristics = pd.DataFrame(rows)
characteristics.to_csv(OUT / "characteristics.csv", index=False, encoding="utf-8-sig")
prev = pd.DataFrame(prevalence_rows("MIMIC-IV", main, lat[main].to_numpy(float), race_index(lat.race6), C[main].to_numpy(float), race_index(C.race6)))
prev.to_csv(OUT / "prevalence.csv", index=False, encoding="utf-8-sig")
characteristics[["group", "patients", "ecgs", "age", "female_pct", "prev_Composite SHD", "prev_LVEF ≤45%", "prev_Aortic stenosis", "pos_Aortic stenosis",
                 "gap_days"]].round(1)

## 3.6 Conversion to the EchoNext format: constants

The MIMIC-IV waveforms are converted to the format of the EchoNext release before they enter the models, so that both cohorts pass through identical pipelines: resampling from 500 to 250 Hz, a median filter of each lead, clipping of each lead at its 0.1st and 99.9th percentiles, and standardisation of each lead with dataset-wide constants. The EchoNext release does not state the median kernel; it is estimated here.

* **Median kernel.** A signal median-filtered with kernel *k* hardly changes when filtered again with a kernel ≤ *k*, but changes under larger kernels. The fraction of samples changed by kernels 3, 5 and 7 in 1,500 EchoNext ECGs is compared with the same profile of 1,500 MIMIC-IV ECGs filtered with kernel 1 (none), 3, 5 or 7; the closest profile gives the kernel (3).
* **Clipping bounds and standardisation constants**: per lead, from 10,000 randomly chosen MIMIC-IV ECGs (after resampling and filtering).

The constants are saved to `RESULTS_DIR/03_mimic/format_stats.json` and used in notebook 04.

In [ ]:
from scipy import signal as ss
from scipy.ndimage import median_filter
from scipy.signal import medfilt

# EchoNext reference profile: fraction of interior samples changed by a median filter of length k
xe = np.ascontiguousarray(np.asarray(echonext_waveforms("val")[:1500], dtype=np.float64)[:, 0].transpose(0, 2, 1))     # 1500 x 12 x 2500
ref = {}
for k in (3, 5, 7):
    dd = np.abs(medfilt(xe, kernel_size=(1, 1, k)) - xe)[..., k:-k]
    ref[k] = float((dd > 1e-6).mean())
del xe


def to_250(x):
    return ss.resample_poly(x, 1, 2, axis=-1).astype(np.float32)               # anti-aliased 500 -> 250 Hz


def med(y, k):
    return median_filter(y, size=(1, 1, k), mode="nearest") if k > 1 else y


def frac_changed(y, k):
    dd = np.abs(med(y, k) - y)[..., k:-k]
    return float((dd > 1e-6 * y.std()).mean())


ok_rows = np.flatnonzero(pd.read_parquet(WAVE_QC).wave_ok.to_numpy())
Wv = np.load(WAVE, mmap_mode="r")
rng = np.random.default_rng(0)
target = np.array([ref[k] for k in (3, 5, 7)])
s0 = int(rng.integers(0, len(ok_rows) - 1500))
y = to_250(np.asarray(Wv[ok_rows[s0:s0 + 1500]]).astype(np.float32) / GAIN)
prof = {k0: [frac_changed(med(y, k0), k) for k in (3, 5, 7)] for k0 in (1, 3, 5, 7)}
kernel = min(prof, key=lambda k0: float(np.abs(np.array(prof[k0]) - target).sum()))
pick = np.sort(rng.choice(ok_rows, 10_000, replace=False))
Yr = np.concatenate([med(to_250(np.asarray(Wv[pick[i:i + 1000]]).astype(np.float32) / GAIN), kernel) for i in range(0, len(pick), 1000)])
lo, hi = np.percentile(Yr, 0.1, axis=(0, 2)), np.percentile(Yr, 99.9, axis=(0, 2))
Yc = np.clip(Yr, lo[None, :, None], hi[None, :, None])
fmt = {"median_kernel": int(kernel), "echonext_profile_k3_5_7": target.tolist(), "mimic_profile_by_kernel": {str(k): v for k, v in prof.items()},
       "n_sample": int(len(pick)), "leads": LEADS, "clip_lo_mv": lo.tolist(), "clip_hi_mv": hi.tolist(), "mean_mv": Yc.mean((0, 2)).tolist(),
       "sd_mv": Yc.std((0, 2)).tolist()}
save_json(fmt, OUT / "format_stats.json")
del Yr, Yc, y
print("median kernel:", kernel)
pd.DataFrame({"clip low (mV)": lo, "clip high (mV)": hi, "mean (mV)": fmt["mean_mv"], "SD (mV)": fmt["sd_mv"]}, index=LEADS).round(3)